CONNECTING THE ESSENTIALS

In [22]:
from langchain_chroma import Chroma
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from dotenv import load_dotenv
import os
load_dotenv()
groq_api_key = os.getenv("GROQ_API_KEY")
gemini_api_key = os.getenv("GEMINI_API_KEY")
vectorstore=Chroma(persist_directory="./chroma_db",collection_name="abc_company",embedding_function=GoogleGenerativeAIEmbeddings(model="gemini-embedding-001",google_api_key=gemini_api_key))
print("Connected to ChromaDB")

Connected to ChromaDB


CREATING TOOLS(RAG SEARCH, GENERAL GOOGLE SEARCH, RESEARCH PAPER RELATED QUERIES)

In [23]:
from langchain_core.tools import tool
from ddgs import DDGS
import arxiv

@tool
def rag_search(question: str) -> str:
    """Search the ABC Company document for relevant information."""

    results = vectorstore.similarity_search(
        question,
        k=3
    )

    context = "\n\n".join(
        doc.page_content
        for doc in results
    )

    return context

@tool
def web_search(query: str) -> str:
    """Search the web for general and current information."""

    results = DDGS().text(
        query,
        max_results=5
    )

    output = []

    for result in results:
        output.append(
            f"Title: {result['title']}\n"
            f"URL: {result['href']}\n"
            f"Description: {result['body']}"
        )

    return "\n\n".join(output)

@tool
def arxiv_search(query: str) -> str:
    """Search arXiv for research papers."""

    client = arxiv.Client()

    search = arxiv.Search(
        query=query,
        max_results=5,
        sort_by=arxiv.SortCriterion.Relevance
    )

    results = client.results(search)

    output = []

    for paper in results:
        authors = ", ".join(
            author.name
            for author in paper.authors
        )

        output.append(
            f"Title: {paper.title}\n"
            f"Authors: {authors}\n"
            f"URL: {paper.entry_id}\n"
            f"Summary: {paper.summary}"
        )

    return "\n\n".join(output)

BINDING THE TOOLS

In [24]:
from langchain_groq import ChatGroq
llm=ChatGroq(model="openai/gpt-oss-20b",api_key=groq_api_key,temperature=0)
tools = [
    rag_search,
    web_search,
    arxiv_search
]
llm_with_tools=llm.bind_tools(tools)
print("LLM is successfully connected with tools")


LLM is successfully connected with tools


REACT AGENT CREATION AND A SAMPLE RUN

In [25]:
from langchain.agents import create_agent
my_agent=create_agent(llm_with_tools,tools)
response=my_agent.invoke({"messages": [{"role": "system", "content": "You are a helpful assistant."}, {"role": "user", "content": "What is the weather in Chennai today?"}]})
print(response["messages"][-1].content)

**Chennai – Weather (as of today, 13 July 2026)**  

| Item | Detail |
|------|--------|
| **Current temperature** | ~ 35 °C (high of 37 °C expected for the day) |
| **Feels‑like** | ~ 36 °C (due to 79 % humidity) |
| **Sky** | Overcast / cloudy |
| **Wind** | Light, 5 mph (≈ 9 km/h) from the northeast |
| **Humidity** | 79 % |
| **Precipitation** | Minimal chance of rain (very low) |
| **General outlook** | Hot and humid with overcast skies; comfortable for indoor activities but stay hydrated if outdoors. |

**Source** – LatestLY weather update for Chennai, 13 July 2026: “Chennai Weather Forecast & Update for Today, Monday, 13 July 2026 – Overcast Skies and High Humidity, High of 37 °C”【7†L1-L5】.


REACT AGENT WITH TEMPORARY MEMORY

In [26]:
from pydantic import BaseModel, Field, HttpUrl


class ResearchPaper(BaseModel):
    title: str = Field(min_length=1)
    link: HttpUrl


conversation_responses = []


def run_conversation():
    paper_keywords = (
        "paper", "papers", "research", "arxiv", "academic",
        "publication", "study", "studies"
    )

    while True:
        question = input("\nQuestion: ").strip()
        if not question:
            print("Please enter a question.")
            continue

        print(f"\nQuestion: {question}")

        if any(keyword in question.casefold() for keyword in paper_keywords):
            try:
                search = arxiv.Search(
                    query=question,
                    max_results=5,
                    sort_by=arxiv.SortCriterion.Relevance
                )
                papers = [
                    ResearchPaper(title=paper.title.strip(), link=paper.entry_id)
                    for paper in arxiv.Client().results(search)
                ]

                if papers:
                    answer = "\n\n".join(
                        f"Title: {paper.title}\nLink: {paper.link}"
                        for paper in papers
                    )
                else:
                    answer = "No research papers were found for that query."
            except Exception as error:
                answer = f"Research paper search or validation failed: {error}"
        else:
            result = my_agent.invoke({"messages": [{"role": "user", "content": question}]})
            answer = result["messages"][-1].content

        print(f"\nAnswer:\n{answer}")
        conversation_responses.append(f"Question: {question}\nAnswer: {answer}")

        if input("\nDo you want to continue? (yes/no): ").strip().casefold() == "no":
            break

    print(f"\nStored {len(conversation_responses)} responses in conversation_responses.")





In [27]:
run_conversation()


Question: Who is the current Chief Minister of Tamil Nadu?

Answer:
The current Chief Minister of Tamil Nadu is **C. Joseph Vijay**. He was sworn in as the 22nd Chief Minister on 10 May 2026 following the 2026 Tamil Nadu Legislative Assembly election.

Question: Research papers on Nanomaterials

Answer:
Title: Toxicity of Carbon Nanomaterials
Link: http://arxiv.org/abs/2109.13918v1

Title: On the Categorization of Nanomaterials
Link: http://arxiv.org/abs/2012.15742v1

Title: A Perspective on the Grouping and Categorization of Nanomaterials
Link: http://arxiv.org/abs/2101.03869v1

Title: Categorization of Next Generation Nanomaterials: International Cooperation on the Categorization of Nanomaterials in the Regulatory Chemical Context
Link: http://arxiv.org/abs/2101.03870v1

Title: Application of carbon nanomaterials in the electronic industry
Link: http://arxiv.org/abs/2004.11629v1

Stored 2 responses in conversation_responses.
